In order to analyse the dataset successfully, I need to understand the connections between the tables. Now, I will connect them.

In [ ]:
import pandas as pd



files = {
    "dictionary": "C:/Projekt_Ordner/Sales_Analysis/Data/Cleaned_Data/1_maven_fuzzy_factory_data_dictionary_clean.csv",
    "order_item_refunds": "C:/Projekt_Ordner/Sales_Analysis/Data/Cleaned_Data/2_order_item_refunds_clean.csv",
    "order_items": "C:/Projekt_Ordner/Sales_Analysis/Data/Cleaned_Data/3_order_items_clean.csv",
    "orders": "C:/Projekt_Ordner/Sales_Analysis/Data/Cleaned_Data/4_orders_clean.csv",
    "products": "C:/Projekt_Ordner/Sales_Analysis/Data/Cleaned_Data/5_products_clean.csv",
    "website_pageviews": "C:/Projekt_Ordner/Sales_Analysis/Data/Cleaned_Data/6_website_pageviews_clean.csv",
    "website_sessions": "C:/Projekt_Ordner/Sales_Analysis/Data/Cleaned_Data/7_website_sessions_clean.csv"
}

dfs = {}
for name, path in files.items():
    dfs[name] = pd.read_csv(path)
    print(f"{name}")
    print(list(dfs[name].columns))
    print()


dictionary
['Table', 'Field', 'Description']

order_item_refunds
['order_item_refund_id', 'created_at', 'order_item_id', 'order_id', 'refund_amount_usd']

order_items
['order_item_id', 'created_at', 'order_id', 'product_id', 'is_primary_item', 'price_usd', 'cogs_usd']

orders
['order_id', 'created_at', 'website_session_id', 'user_id', 'primary_product_id', 'items_purchased', 'price_usd', 'cogs_usd']

products
['product_id', 'created_at', 'product_name']

website_pageviews
['website_pageview_id', 'created_at', 'website_session_id', 'pageview_url']

website_sessions
['website_session_id', 'created_at', 'user_id', 'is_repeat_session', 'utm_source', 'utm_campaign', 'utm_content', 'device_type', 'http_referer']



Columns which appear in more than one file are the key joins.<br>
These are: order_item_id; order_id; created_at; product_id; price_usd; cogs_usd; website_session_id; user_id <br>
<br>

__Now I know the key joins and build the master table.__


I connect the key column "order_id" with the file "order_item_refunds" and "oder_items"; the key columns "created_at" with all files;"product_id" with  "order_items", "orders" and "products"; "price_usd" with "order_items" and "orders"; "cogs_usd" with "order_items" and "orders"; "website_session_id" with "website_sessions", "website_pageviews" and "orders"; and "user_id" with "orders" and "website_sessions".

In [18]:
dictionary = pd.read_csv("C:/Projekt_Ordner/Sales_Analysis/Data/Cleaned_Data/1_maven_fuzzy_factory_data_dictionary_clean.csv")
order_item_refunds = pd.read_csv("C:/Projekt_Ordner/Sales_Analysis/Data/Cleaned_Data/2_order_item_refunds_clean.csv")
order_items = pd.read_csv("C:/Projekt_Ordner/Sales_Analysis/Data/Cleaned_Data/3_order_items_clean.csv")
orders = pd.read_csv("C:/Projekt_Ordner/Sales_Analysis/Data/Cleaned_Data/4_orders_clean.csv")
products = pd.read_csv("C:/Projekt_Ordner/Sales_Analysis/Data/Cleaned_Data/5_products_clean.csv")
website_pageviews = pd.read_csv("C:/Projekt_Ordner/Sales_Analysis/Data/Cleaned_Data/6_website_pageviews_clean.csv")
website_sessions = pd.read_csv("C:/Projekt_Ordner/Sales_Analysis/Data/Cleaned_Data/7_website_sessions_clean.csv")


# To convert the date columns

for df in [order_item_refunds, order_items, orders, products, website_pageviews, website_sessions]:
    df["created_at"] = pd.to_datetime(df["created_at"])


# Master table 1 - Sales

df_sales = pd.merge(order_items, orders[["order_id", "created_at", "website_session_id"]], 
                    on="order_id", how="left", suffixes=("", "_order"))   #how=left -> keep all rows from order_items, add matching info from orders


df_sales = pd.merge(df_sales, products[["product_id", "product_name"]], 
                    on="product_id", how="left")


df_sales = pd.merge(df_sales, order_item_refunds[["order_item_id", "refund_amount_usd"]], 
                    on="order_item_id", how="left")


#I add usefull columns for a analysis

df_sales["is_refunded"] = df_sales["refund_amount_usd"].notna()
df_sales["profit_usd"] = df_sales["price_usd"] - df_sales["cogs_usd"]
df_sales["year_month"] = df_sales["created_at"].dt.to_period("M")


print(df_sales.head(3))
print(df_sales.shape)
print("------------------------")


# Master table 2 - Sessions

df_sessions = pd.merge(website_sessions, orders[["order_id", "website_session_id"]], on="website_session_id", how="left")

df_sessions["converted"] = df_sessions["order_id"].notna()

print(df_sessions.head(3))

print(df_sessions.shape)



   order_item_id          created_at  order_id  product_id  is_primary_item  \
0              1 2012-03-19 10:42:46         1           1                1   
1              2 2012-03-19 19:27:37         2           1                1   
2              3 2012-03-20 06:44:45         3           1                1   

   price_usd  cogs_usd    created_at_order  website_session_id  \
0      49.99     19.49 2012-03-19 10:42:46                  20   
1      49.99     19.49 2012-03-19 19:27:37                 104   
2      49.99     19.49 2012-03-20 06:44:45                 147   

             product_name  refund_amount_usd  is_refunded  profit_usd  \
0  The Original Mr. Fuzzy                NaN        False        30.5   
1  The Original Mr. Fuzzy                NaN        False        30.5   
2  The Original Mr. Fuzzy                NaN        False        30.5   

  year_month  
0    2012-03  
1    2012-03  
2    2012-03  
(40025, 14)
------------------------
   website_session_id       

Saving the dfs

In [19]:
df_sales.to_csv("C:/Projekt_Ordner/Sales_Analysis/Data/Cleaned_Data/df_sales.csv", index=False)
df_sessions.to_csv("C:/Projekt_Ordner/Sales_Analysis/Data/Cleaned_Data/df_sessions.csv", index=False)


Short Test

In [15]:
# Monthly Revenue

print(df_sales.groupby("year_month") ["price_usd"].sum())

print("--------------")
#Refund Rate

print(df_sales["is_refunded"].mean())


year_month
2012-03      2999.40
2012-04      4949.01
2012-05      5398.92
2012-06      6998.60
2012-07      8448.31
2012-08     11397.72
2012-09     14347.13
2012-10     18546.29
2012-11     30893.82
2012-12     25294.94
2013-01     19966.10
2013-02     26515.02
2013-03     19896.15
2013-04     28584.47
2013-05     29364.29
2013-06     30544.07
2013-07     31143.96
2013-08     31373.92
2013-09     32723.65
2013-10     38242.62
2013-11     46631.02
2013-12     58262.60
2014-01     56568.89
2014-02     66012.52
2014-03     68189.73
2014-04     78725.43
2014-05     88935.27
2014-06     80051.25
2014-07     83288.55
2014-08     84716.08
2014-09     92232.49
2014-10    103905.98
2014-11    128162.98
2014-12    144823.02
2015-01    132211.54
2015-02    129212.94
2015-03     78951.07
Freq: M, Name: price_usd, dtype: float64
--------------
0.043247970018738285
